In [25]:
#Model Training

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam

import os

In [27]:
df = pd.read_csv("../data/processed_data/feature_engineered_dataset.csv")

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [28]:
X = df.drop("Heart_Disease", axis=1)
y = df["Heart_Disease"]

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [30]:
# ---------------------------------------------
# Feature Scaling using RobustScaler
# ---------------------------------------------

scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("="*50)
print("Robust Scaling Completed")
print("="*50)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Robust Scaling Completed
Training Shape : (54428, 14)
Testing Shape : (13608, 14)


In [31]:
# ==========================================================
# STEP 5 - HYPERPARAMETER TUNING
# Function to create ANN models
# ==========================================================

def create_ann(
    input_features,
    neurons=(128, 64, 32),
    dropout_rate=0.25,
    learning_rate=0.0005
):
    
    model = Sequential()

    # Input Layer
    model.add(Input(shape=(input_features,)))

    # Hidden Layer 1
    model.add(Dense(neurons[0], activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Hidden Layer 2
    model.add(Dense(neurons[1], activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Hidden Layer 3
    model.add(Dense(neurons[2], activation="relu"))

    # Output Layer
    model.add(Dense(1, activation="sigmoid"))

    # Compile
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("ANN model creation function is ready.")

ANN model creation function is ready.


In [32]:
# ==========================================================
# Training Callbacks
# ==========================================================

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

print("Training callbacks are ready.")

Training callbacks are ready.


In [33]:
# ==========================================================
# Hyperparameter Configurations
# ==========================================================

experiments = [
    
    {
        "name": "Experiment 1",
        "neurons": (128, 64, 32),
        "dropout": 0.20,
        "learning_rate": 0.0005,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 2",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 3",
        "neurons": (128, 64, 32),
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 4",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.001,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 5",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.00025,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 6",
        "neurons": (256, 128, 64),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 32
    },
    
    {
        "name": "Experiment 7",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 64
    }
]

print("Number of experiments:", len(experiments))

Number of experiments: 7


In [34]:
# ==========================================================
# Run Hyperparameter Experiments
# ==========================================================

results = []

for experiment in experiments:

    print("\n" + "=" * 70)
    print(experiment["name"])
    print("=" * 70)

    # Create model
    model = create_ann(
        input_features=X_train.shape[1],
        neurons=experiment["neurons"],
        dropout_rate=experiment["dropout"],
        learning_rate=experiment["learning_rate"]
    )

    # Create fresh callbacks for each experiment
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=0
    )

    # Train model
    history = model.fit(
        X_train,
        y_train,
        validation_split=0.20,
        epochs=100,
        batch_size=experiment["batch_size"],
        callbacks=[
            early_stopping,
            reduce_lr
        ],
        verbose=0
    )

    # Predict test data
    y_prob = model.predict(
        X_test,
        verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    results.append({
        "Experiment": experiment["name"],
        "Neurons": str(experiment["neurons"]),
        "Dropout": experiment["dropout"],
        "Learning Rate": experiment["learning_rate"],
        "Batch Size": experiment["batch_size"],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    })

    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC AUC   : {roc_auc:.4f}")


Experiment 1
Accuracy  : 73.51%
Precision : 0.7505
Recall    : 0.6931
F1 Score  : 0.7207
ROC AUC   : 0.8009

Experiment 2
Accuracy  : 73.24%
Precision : 0.7538
Recall    : 0.6788
F1 Score  : 0.7144
ROC AUC   : 0.8020

Experiment 3
Accuracy  : 73.27%
Precision : 0.7458
Recall    : 0.6944
F1 Score  : 0.7192
ROC AUC   : 0.8017

Experiment 4
Accuracy  : 73.40%
Precision : 0.7489
Recall    : 0.6927
F1 Score  : 0.7197
ROC AUC   : 0.8020

Experiment 5
Accuracy  : 73.38%
Precision : 0.7507
Recall    : 0.6889
F1 Score  : 0.7185
ROC AUC   : 0.8012

Experiment 6
Accuracy  : 73.44%
Precision : 0.7543
Recall    : 0.6842
F1 Score  : 0.7175
ROC AUC   : 0.8026

Experiment 7
Accuracy  : 73.33%
Precision : 0.7444
Recall    : 0.6992
F1 Score  : 0.7211
ROC AUC   : 0.8015


In [35]:
# ==========================================================
# Compare All Experiments
# ==========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df.reset_index(drop=True, inplace=True)

results_df

,Experiment,Neurons,Dropout,Learning Rate,Batch Size,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Experiment 1,"(128, 64, 32)",0.20,0.00050,32,0.735082,0.750484,0.693099,0.720651,0.800860
1,Experiment 6,"(256, 128, 64)",0.25,0.00050,32,0.734421,0.754314,0.684156,0.717524,0.802645
2,Experiment 4,"(128, 64, 32)",0.25,0.00100,32,0.733980,0.748912,0.692652,0.719684,0.801982
3,Experiment 5,"(128, 64, 32)",0.25,0.00025,32,0.733833,0.750690,0.688925,0.718483,0.801190
4,Experiment 7,"(128, 64, 32)",0.25,0.00050,64,0.733319,0.744367,0.699210,0.721082,0.801474
5,Experiment 3,"(128, 64, 32)",0.30,0.00050,32,0.732657,0.745798,0.694440,0.719203,0.801729
6,Experiment 2,"(128, 64, 32)",0.25,0.00050,32,0.732363,0.753849,0.678790,0.714353,0.802011


In [36]:
# ==========================================================
# Best Hyperparameter Configuration
# ==========================================================

best_result = results_df.iloc[0]

print("=" * 60)
print("BEST HYPERPARAMETER CONFIGURATION")
print("=" * 60)

print("Experiment       :", best_result["Experiment"])
print("Neurons          :", best_result["Neurons"])
print("Dropout          :", best_result["Dropout"])
print("Learning Rate    :", best_result["Learning Rate"])
print("Batch Size       :", best_result["Batch Size"])

print("\nPerformance:")
print("Accuracy         :", f"{best_result['Accuracy'] * 100:.2f}%")
print("Precision        :", f"{best_result['Precision']:.4f}")
print("Recall           :", f"{best_result['Recall']:.4f}")
print("F1 Score         :", f"{best_result['F1 Score']:.4f}")
print("ROC AUC          :", f"{best_result['ROC AUC']:.4f}")

BEST HYPERPARAMETER CONFIGURATION
Experiment       : Experiment 1
Neurons          : (128, 64, 32)
Dropout          : 0.2
Learning Rate    : 0.0005
Batch Size       : 32

Performance:
Accuracy         : 73.51%
Precision        : 0.7505
Recall           : 0.6931
F1 Score         : 0.7207
ROC AUC          : 0.8009


In [37]:
# ==========================================================
# Train Final Best Model
# ==========================================================

best_neurons = eval(best_result["Neurons"])
best_dropout = float(best_result["Dropout"])
best_learning_rate = float(best_result["Learning Rate"])
best_batch_size = int(best_result["Batch Size"])

best_model = create_ann(
    input_features=X_train.shape[1],
    neurons=best_neurons,
    dropout_rate=best_dropout,
    learning_rate=best_learning_rate
)

best_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

best_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

best_history = best_model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=best_batch_size,
    callbacks=[
        best_early_stopping,
        best_reduce_lr
    ],
    verbose=1
)

print("\nFinal best model training completed.")

Epoch 1/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.7117 - loss: 0.5798 - val_accuracy: 0.7293 - val_loss: 0.5470 - learning_rate: 5.0000e-04
Epoch 2/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7250 - loss: 0.5594 - val_accuracy: 0.7310 - val_loss: 0.5437 - learning_rate: 5.0000e-04
Epoch 3/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7255 - loss: 0.5570 - val_accuracy: 0.7297 - val_loss: 0.5441 - learning_rate: 5.0000e-04
Epoch 4/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.7271 - loss: 0.5534 - val_accuracy: 0.7294 - val_loss: 0.5447 - learning_rate: 5.0000e-04
Epoch 5/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7305 - loss: 0.5520 - val_accuracy: 0.7302 - val_loss: 0.5443 - learning_rate: 5.0000e-04
Epoch 6/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.7300 - loss: 0.5527 - val_accuracy: 0.7325 - val_loss: 0.5424 - learning_rate: 5.0000e-04
Epoch 7/100
1361/1361 ━━━━━━━━━━━━━━━━━━━━ 

In [38]:
# ==========================================================
# Final Model Evaluation
# ==========================================================

y_prob = best_model.predict(
    X_test,
    verbose=0
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

final_accuracy = accuracy_score(y_test, y_pred)
final_precision = precision_score(y_test, y_pred)
final_recall = recall_score(y_test, y_pred)
final_f1 = f1_score(y_test, y_pred)
final_auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("FINAL TUNED ANN RESULTS")
print("=" * 60)

print("Accuracy  :", f"{final_accuracy * 100:.2f}%")
print("Precision :", f"{final_precision:.4f}")
print("Recall    :", f"{final_recall:.4f}")
print("F1 Score  :", f"{final_f1:.4f}")
print("ROC AUC   :", f"{final_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

FINAL TUNED ANN RESULTS
Accuracy  : 73.48%
Precision : 0.7579
Recall    : 0.6789
F1 Score  : 0.7163
ROC AUC   : 0.8027

Confusion Matrix:
[[5444 1455]
 [2154 4555]]


In [39]:
# ==========================================================
# Save Final Tuned Model
# ==========================================================

import os
import joblib

os.makedirs("../models", exist_ok=True)

best_model.save(
    "../models/heart_disease_ann_tuned.keras"
)

joblib.dump(
    scaler,
    "../models/robust_scaler.pkl"
)

print("Final tuned ANN model saved successfully.")
print("Model  : ../models/heart_disease_ann_tuned.keras")
print("Scaler : ../models/robust_scaler.pkl")

Final tuned ANN model saved successfully.
Model  : ../models/heart_disease_ann_tuned.keras
Scaler : ../models/robust_scaler.pkl
